# Required tasks — Topics & Sentiment

Deck slides 3–4. Topic taxonomy with examples; sentiment as a multi-resolution signal.

In [ ]:
from ti.enrich.taxonomy import load as load_taxonomy
from ti.store import duckdb_store
import pandas as pd
con = duckdb_store.connect()

## Topic taxonomy (Opus-bootstrapped, frozen v1)

Named, actionable themes — not `cluster_3`. Each subtopic carries a description grounded in the corpus's own language.

In [ ]:
tax = load_taxonomy()
for theme in tax.themes:
    print(f'\n{theme.name}: {theme.description}')
    for s in theme.subtopics:
        print(f'   - {s.name}: {s.description[:80]}')

## Topic frequency by theme

In [ ]:
con.execute('SELECT theme, COUNT(*) AS n_classifications, COUNT(DISTINCT meeting_id) AS distinct_meetings FROM topics GROUP BY theme ORDER BY n_classifications DESC').df()

## Sentiment by call type — *as a signal, not a score*

- Average meeting score (pre-tagged 1–5)
- Average arc slope: positive = call ends better than it started; negative = call deteriorates

In [ ]:
con.execute("""
  SELECT call_type,
         COUNT(*) AS n,
         ROUND(AVG(sentiment_score),2) AS avg_score,
         ROUND(AVG(arc_slope),3) AS avg_slope,
         SUM(CASE WHEN arc_slope > 0 THEN 1 ELSE 0 END) AS calls_that_improved,
         SUM(CASE WHEN arc_slope < 0 THEN 1 ELSE 0 END) AS calls_that_degraded
  FROM transcripts
  GROUP BY call_type
  ORDER BY avg_score
""").df()

## Per-product sentiment (when product mentioned)

In [ ]:
con.execute("""
  SELECT entity_name AS product,
         COUNT(*) AS meetings_mentioning,
         SUM(mention_count) AS total_mentions,
         ROUND(AVG(avg_sentiment),2) AS avg_sentiment_when_mentioned
  FROM entity_sentiment
  WHERE entity_kind='product'
  GROUP BY entity_name
  ORDER BY total_mentions DESC
""").df()

## Disagreement vs pre-tagged sentiment (sanity check)

In [ ]:
con.execute("""
  SELECT sentiment_label AS pre_tag,
         ROUND(AVG(arc_slope),3) AS our_avg_arc_slope,
         COUNT(*) AS n
  FROM transcripts
  GROUP BY sentiment_label
  ORDER BY our_avg_arc_slope DESC
""").df()